# Overfitting to the Dev Set

**Feel it, then fix it.**

You'll iterate on a judge prompt, watch the dev score climb, then see the held-out gap.
The dev score going up *feels* like real progress. The held-out gap is the gut punch.

**The key mechanism:** you *see* the dev data. You read the examples, understand the
patterns, and encode them into the prompt — including copy-pasting dev examples as
few-shot. This is **information leakage**: the data directly shapes the system.

## Setup

- **Task:** LLM judge scoring customer-support responses 1-5 on Correctness
- **Dev set:** 30 examples (you iterate on these)
- **Held-out test set:** 100 examples (you touch this ONCE at the end)
- Synthetic data, so the lab runs without any external dataset

In [ ]:
import json
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
MODEL = "gpt-4.1-mini"

CACHE_PATH = Path("overfitting_data_cache.json")

print("model:", MODEL)
print("api key:", "ok" if os.environ.get("OPENAI_API_KEY") else "MISSING")
print("cache:", "found" if CACHE_PATH.exists() else "will generate")

## Step 1 — Synthetic customer-support data

We create realistic exchanges with known quality levels. Each example has a query,
a response, and a ground-truth `true_score` (1-5).

If a cache exists we reuse it — that way you can re-run the lab deterministically.

In [ ]:
GENERATE_PROMPT = """Generate a realistic customer support exchange for an e-commerce company.

Return JSON with these fields:
- "query": the customer's message (1-3 sentences)
- "response": the support agent's response (2-5 sentences)
- "true_score": Correctness score 1-5 (how factually correct and complete the response is)
- "reasoning": why this score (1 sentence)

For variety, generate exchanges across this quality spectrum:
- Score 1-2: wrong information, misunderstands the issue, or gives irrelevant answer
- Score 3: partially correct but missing key details
- Score 4: mostly correct with minor gaps
- Score 5: fully correct and complete

Target score: {target_score}

Return ONLY valid JSON, no markdown."""


def generate_examples(n, seed=42):
    random.seed(seed)
    scores = []
    for _ in range(n):
        r = random.random()
        if r < 0.15:
            scores.append(random.choice([1, 2]))
        elif r < 0.35:
            scores.append(3)
        elif r < 0.70:
            scores.append(4)
        else:
            scores.append(5)

    print(f"generating {n} examples...")
    examples = []
    for i, target in enumerate(scores):
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": GENERATE_PROMPT.format(target_score=target)}],
            temperature=0.9,
            response_format={"type": "json_object"},
        )
        try:
            ex = json.loads(resp.choices[0].message.content)
            ex["id"] = i
            examples.append(ex)
        except json.JSONDecodeError:
            continue
        if (i + 1) % 10 == 0:
            print(f"  {i + 1}/{n}")
    return examples


In [ ]:
if CACHE_PATH.exists():
    cached = json.loads(CACHE_PATH.read_text())
    dev_set, test_set = cached["dev"], cached["test"]
    print(f"loaded cache: dev={len(dev_set)}, test={len(test_set)}")
else:
    all_examples = generate_examples(130)
    random.shuffle(all_examples)
    dev_set = all_examples[:30]
    test_set = all_examples[30:]
    for i, ex in enumerate(dev_set): ex["id"] = f"dev_{i}"
    for i, ex in enumerate(test_set): ex["id"] = f"test_{i}"
    CACHE_PATH.write_text(json.dumps({"dev": dev_set, "test": test_set}, indent=2))
    print(f"generated and cached: dev={len(dev_set)}, test={len(test_set)}")

In [ ]:
# Peek at one example
ex = dev_set[0]
print("query:   ", ex["query"])
print("response:", ex["response"])
print("score:   ", ex["true_score"], "—", ex["reasoning"])

## Step 2 — The judge prompt (v0 baseline)

This is the prompt we'll iterate on. It asks the LLM to score a support response
on Correctness (1-5). Start simple, we'll improve it step by step.

In [ ]:
JUDGE_V0 = """Rate the following customer support response for Correctness on a scale of 1-5.

1 = Completely wrong or irrelevant
2 = Mostly wrong with some correct elements
3 = Partially correct but missing key details
4 = Mostly correct with minor gaps
5 = Fully correct and complete

Customer query: {query}

Agent response: {response}

Return JSON with:
- "score": integer 1-5
- "reasoning": brief explanation (1-2 sentences)

Return ONLY valid JSON."""


def run_judge(prompt_template, examples, label=""):
    """Run the judge on a list of examples. Returns per-item results + aggregate stats."""
    results = []
    start = time.time()
    for ex in examples:
        prompt = prompt_template.format(query=ex["query"], response=ex["response"])
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            response_format={"type": "json_object"},
        )
        try:
            parsed = json.loads(resp.choices[0].message.content)
            judge_score = int(parsed.get("score", 0))
        except (json.JSONDecodeError, ValueError):
            judge_score = 0
        results.append({
            "id": ex["id"],
            "true_score": ex["true_score"],
            "judge_score": judge_score,
            "exact_match": judge_score == ex["true_score"],
            "within_1": abs(judge_score - ex["true_score"]) <= 1,
        })
    elapsed = time.time() - start
    exact = sum(r["exact_match"] for r in results) / len(results) * 100
    within1 = sum(r["within_1"] for r in results) / len(results) * 100
    mae = sum(abs(r["judge_score"] - r["true_score"]) for r in results) / len(results)
    print(f"[{label}] n={len(results)}  exact={exact:.1f}%  within1={within1:.1f}%  MAE={mae:.2f}  ({elapsed:.1f}s)")
    return {"results": results, "exact": exact, "within1": within1, "mae": mae, "elapsed": elapsed}


In [ ]:
v0_dev = run_judge(JUDGE_V0, dev_set, label="v0 DEV")

## Step 3 — v1: a more detailed rubric

Look at dev-set errors and write a richer rubric. This *should* generalize — it's
just better instructions, not leakage.

In [ ]:
print("dev errors (v0):")
for r in v0_dev["results"]:
    if not r["exact_match"]:
        ex = next(e for e in dev_set if e["id"] == r["id"])
        print(f"  [{r['id']}] true={r['true_score']} judge={r['judge_score']}  {ex['query'][:60]}...")

In [ ]:
JUDGE_V1 = """You are an expert evaluator for customer support quality.

Rate the following customer support response for **Correctness** on a scale of 1-5.

## Rubric
- **1 (Wrong):** The response contains factually incorrect information, misidentifies the
  customer's issue, or provides an irrelevant answer. The customer would be misled.
- **2 (Mostly wrong):** The response addresses the topic but contains significant errors
  or misunderstandings. Some elements may be correct but the overall answer is unreliable.
- **3 (Partially correct):** The response gets the general idea right but is missing key
  details, caveats, or steps. The customer would need to follow up for a complete answer.
- **4 (Mostly correct):** The response is accurate on all major points with only minor
  gaps or imprecisions. The customer could act on this information successfully.
- **5 (Fully correct):** The response is factually accurate, complete, and directly
  addresses the customer's specific question. No follow-up needed.

## Important
- Focus on factual correctness, not tone or politeness.
- A response that is polite but wrong should score low.
- A response that is brusque but accurate should score high on Correctness.

## Exchange
Customer: {query}
Agent: {response}

Return JSON with "score" (integer 1-5) and "reasoning" (1-2 sentences).
Return ONLY valid JSON."""

v1_dev = run_judge(JUDGE_V1, dev_set, label="v1 DEV")

## Step 4 — v2: few-shot examples pulled from dev (the leakage move)

Now we copy-paste examples from the dev set into the prompt as "calibration examples."
This is the canonical overfitting pattern. The prompt now *literally contains* items
from the evaluation set.

In [ ]:
errors_v1 = [(r, next(e for e in dev_set if e["id"] == r["id"]))
             for r in v1_dev["results"] if not r["exact_match"]]

fewshot_examples = [
    {"query": ex["query"], "response": ex["response"], "score": ex["true_score"],
     "reasoning": f"Ground truth score is {ex['true_score']}."}
    for _, ex in errors_v1[:3]
]

if len(fewshot_examples) < 3:
    correct = [(r, next(e for e in dev_set if e["id"] == r["id"]))
               for r in v1_dev["results"] if r["exact_match"]]
    for _, ex in correct[:3 - len(fewshot_examples)]:
        fewshot_examples.append({
            "query": ex["query"], "response": ex["response"],
            "score": ex["true_score"],
            "reasoning": f"Ground truth score is {ex['true_score']}.",
        })

fewshot_block = "\n\n".join(
    f"### Example {i+1}\nCustomer: {fs['query']}\nAgent: {fs['response']}\nScore: {fs['score']}\nReasoning: {fs['reasoning']}"
    for i, fs in enumerate(fewshot_examples)
)

JUDGE_V2 = f"""You are an expert evaluator for customer support quality.

Rate the following customer support response for **Correctness** on a scale of 1-5.

## Rubric
- **1 (Wrong):** Factually incorrect, misidentifies the issue, or irrelevant answer.
- **2 (Mostly wrong):** Addresses the topic but significant errors. Unreliable overall.
- **3 (Partially correct):** General idea right but missing key details. Needs follow-up.
- **4 (Mostly correct):** Accurate on major points, minor gaps. Actionable.
- **5 (Fully correct):** Accurate, complete, directly addresses the question. No follow-up needed.

## Calibration Examples (to set your scale)
{fewshot_block}

## Important
- Focus on factual correctness, not tone or politeness.
- Calibrate your scores to match the examples above.

## Exchange to Rate
Customer: {{query}}
Agent: {{response}}

Return JSON with "score" (integer 1-5) and "reasoning" (1-2 sentences).
Return ONLY valid JSON."""

v2_dev = run_judge(JUDGE_V2, dev_set, label="v2 DEV")

## Step 5 — v3: more leakage, more rules from dev errors

Add *more* dev-derived few-shot and encode dev-specific patterns as "scoring rules."
By construction, these help on dev. Will they help on test?

In [ ]:
errors_v2 = [(r, next(e for e in dev_set if e["id"] == r["id"]))
             for r in v2_dev["results"] if not r["exact_match"]]

additional_examples = [
    {"query": ex["query"], "response": ex["response"], "score": ex["true_score"],
     "reasoning": f"This is a {ex['true_score']} because the response "
                  f"{'addresses' if ex['true_score'] >= 3 else 'fails to address'} "
                  f"the customer's core issue."}
    for _, ex in errors_v2[:3]
]

all_fewshot = fewshot_examples + additional_examples
fewshot_block_v3 = "\n\n".join(
    f"### Example {i+1}\nCustomer: {fs['query']}\nAgent: {fs['response']}\nScore: {fs['score']}\nReasoning: {fs['reasoning']}"
    for i, fs in enumerate(all_fewshot)
)

JUDGE_V3 = f"""You are an expert evaluator for customer support quality.

Rate the following customer support response for **Correctness** on a scale of 1-5.

## Detailed Rubric
- **1 (Wrong):** Factually incorrect, misidentifies the issue, or irrelevant answer.
  Example patterns: gives wrong return policy, confuses products, addresses wrong issue.
- **2 (Mostly wrong):** Addresses the topic but significant errors. Unreliable overall.
  Example patterns: mentions the right product but gives wrong specs or timelines.
- **3 (Partially correct):** General idea right but missing key details. Needs follow-up.
  Example patterns: correct category but skips important caveats or conditions.
- **4 (Mostly correct):** Accurate on major points, minor gaps. Actionable.
  Example patterns: right answer but could mention an additional option or exception.
- **5 (Fully correct):** Accurate, complete, directly addresses the question. No follow-up needed.

## Calibration Examples
{fewshot_block_v3}

## Scoring Rules
- Focus on factual correctness, not tone or politeness.
- If the response gives a correct general answer but misses a specific detail the customer
  asked about, cap at 3.
- If the response mentions a policy or timeline, verify it's consistent with the context.
- Calibrate strictly to the examples above.

## Exchange to Rate
Customer: {{query}}
Agent: {{response}}

Return JSON with "score" (integer 1-5) and "reasoning" (1-2 sentences).
Return ONLY valid JSON."""

v3_dev = run_judge(JUDGE_V3, dev_set, label="v3 DEV")

## Step 6 — the reveal: run all versions on the held-out test set

We touch the 100-example test set **once**, at the end. Now we see how much of the
dev improvement actually generalized.

In [ ]:
v0_test = run_judge(JUDGE_V0, test_set, label="v0 TEST")
v1_test = run_judge(JUDGE_V1, test_set, label="v1 TEST")
v2_test = run_judge(JUDGE_V2, test_set, label="v2 TEST")
v3_test = run_judge(JUDGE_V3, test_set, label="v3 TEST")

## Step 7 — the overfitting trajectory

Two views:
1. A table of dev vs test exact-match per version, plus the gap
2. A chart with two lines: dev climbing, test lagging — the "scissor plot" of overfitting

In [ ]:
versions = [
    ("v0 Baseline",        v0_dev, v0_test),
    ("v1 Rubric",          v1_dev, v1_test),
    ("v2 Few-shot (dev)",  v2_dev, v2_test),
    ("v3 Heavy tuning",    v3_dev, v3_test),
]

print(f"{'Version':<22} {'DEV':>8} {'TEST':>8} {'Gap':>8}")
print("-" * 50)
for name, dev, test in versions:
    gap = dev["exact"] - test["exact"]
    marker = " ←" if gap > 5 else ""
    print(f"{name:<22} {dev['exact']:>7.1f}% {test['exact']:>7.1f}% {gap:>+7.1f}{marker}")

In [ ]:
names = [v[0] for v in versions]
dev_scores = [v[1]["exact"] for v in versions]
test_scores = [v[2]["exact"] for v in versions]
gaps = [d - t for d, t in zip(dev_scores, test_scores)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.plot(names, dev_scores, "o-", linewidth=2.2, markersize=9, color="#2563eb", label="DEV (what you see)")
ax1.plot(names, test_scores, "s-", linewidth=2.2, markersize=9, color="#dc2626", label="TEST (held-out)")
ax1.fill_between(range(len(names)), dev_scores, test_scores, alpha=0.12, color="#dc2626")
ax1.set_ylabel("Exact match (%)")
ax1.set_title("Dev climbs, test lags — the overfitting scissor")
ax1.legend(loc="lower right")
ax1.grid(alpha=0.3)
ax1.set_ylim(40, 90)

bars = ax2.bar(names, gaps, color=["#94a3b8" if g < 5 else "#dc2626" for g in gaps])
ax2.set_ylabel("Dev − Test (pp)")
ax2.set_title("Overfitting gap per version")
ax2.axhline(0, color="black", linewidth=0.8)
ax2.grid(alpha=0.3, axis="y")
for bar, g in zip(bars, gaps):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
             f"{g:+.1f}", ha="center", fontsize=10)

plt.setp(ax1.xaxis.get_majorticklabels(), rotation=15, ha="right")
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=15, ha="right")
plt.tight_layout()
plt.show()

### Per-score confusion: where does the judge break?

For each version, we can compare the distribution of (true, judge) pairs.
This shows *how* it's wrong — tends-to-high, tends-to-low, or just noisy?

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8), sharey=True)
for ax, (name, dev, test) in zip(axes, versions):
    # Use test results (the honest signal)
    errors = [r["judge_score"] - r["true_score"] for r in test["results"]]
    ax.hist(errors, bins=range(-4, 6), rwidth=0.8, color="#dc2626", alpha=0.75, align="left")
    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_title(f"{name}\n(TEST errors)")
    ax.set_xlabel("judge − true")
    ax.set_xticks(range(-4, 5))
axes[0].set_ylabel("count")
plt.tight_layout()
plt.show()

## Discussion

### What did we learn?

Dev climbed steadily — **that felt like progress**. Test barely moved. The
**gap** is the overfitting.

- **v0 → v1** (more detailed rubric): dev moved a few points, test roughly flat
  or slightly down. Within sampling noise for a 30-item dev. What felt like
  progress was mostly noise.
- **v1 → v2** (few-shot copy-pasted from dev): both dev and test jumped. Most of
  the win is genuine (calibration examples really do help) — but a small premium
  on dev is leakage, because the judge now "knows" those specific items.
- **v2 → v3** (more few-shot + dev-derived rules): dev keeps climbing, test
  stalls. This is the **textbook overfit** — encoding dev-specific patterns that
  don't generalize.

### Even the baseline has a gap

v0 dev vs v0 test already differed by several points. That's not overfitting —
it's **sampling noise**. With a 30-item dev, ±5pt swings are routine. The
practical takeaway: don't trust a single-digit "improvement" on a small dev set.

### The fix

1. **Held-out test set you touch ONCE, at the end.**
2. **Track iteration count** — the more you've iterated, the more you should
   discount the dev improvement.
3. **Larger dev sets** are harder to overfit (but not impossible).
4. **Cross-validation** on dev can give you a less noisy signal mid-iteration.

### Does more data help?

Partially. A larger dev set shrinks both the baseline noise AND the overfit gap —
but with enough iterations you'll overfit any finite set. The real fix is
**process discipline** (a strict held-out set), not just more data. More data
buys you time, not safety.